In [0]:
%sql
create or replace view vw_driver_standing_view as
with driver_stats as (
    select
        f.season,
        f.driver_id,
        d.driver_name,
        d.nationality,
        count(*) as race_starts,
        sum(f.points) as total_points,
        sum(case when f.is_win = true then 1 else 0 end) as total_wins,
        sum(case when f.is_podium = true then 1 else 0 end) as total_podiums
    from dev.gold.fact_session_results f
    inner join dev.gold.dim_drivers d
        on f.driver_id = d.driver_id
    where f.session_type = 'race'
    group by all
)
select
    *,
    row_number() over (
        partition by season
        order by total_points desc, total_wins desc, total_podiums desc
    ) as standing_position
from driver_stats

In [0]:
%sql
select * from vw_driver_standing_view